In [1]:
# Nếu chạy Colab L4, cell này chạy 1 lần là đủ.
!pip -q install pandas numpy scikit-learn tqdm torch transformers accelerate bitsandbytes sentence-transformers

In [2]:
!pip install -U huggingface_hub hf_transfer
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [3]:
import os
import re
import json
import random
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from sklearn.metrics import cohen_kappa_score, mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:277: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


In [4]:
# Reproducibility
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [5]:
TRAIN_PATH = "/content/asap_train.csv"
VAL_PATH   = "/content/asap_val.csv"
TEST_PATH  = "/content/asap_test.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

required_cols = ["essay_id", "essay_set", "essay", "domain1_score", "domain1_score_norm"]

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing = set(required_cols) - set(df.columns)
    assert not missing, f"{name} missing columns: {missing}"

train_df.head()

,essay_id,essay_set,essay,domain1_score,domain1_score_norm
0,14876,6,In the excerpt The Mooring Mast by Marcia Amid...,3.0,0.75
1,9985,4,The author concluded this sentence because he ...,0.0,0.00
2,3231,2,"I can think of several books that, I would not...",4.0,0.60
3,21137,8,My best friend @PERSON2 turned thirteen on a b...,39.0,0.65
4,17919,7,A time that I was patient is when I broke my f...,23.0,0.77


In [6]:
ESSAY_SET = 6

p1_train = train_df[train_df["essay_set"] == ESSAY_SET].copy().reset_index(drop=True)
p1_val   = val_df[val_df["essay_set"] == ESSAY_SET].copy().reset_index(drop=True)
p1_test  = test_df[test_df["essay_set"] == ESSAY_SET].copy().reset_index(drop=True)

print("P1 train:", p1_train.shape)
print("P1 val:  ", p1_val.shape)
print("P1 test: ", p1_test.shape)

p1_train[["essay_id", "essay_set", "domain1_score", "domain1_score_norm"]].head()

P1 train: (1261, 5)
P1 val:   (180, 5)
P1 test:  (359, 5)


,essay_id,essay_set,domain1_score,domain1_score_norm
0,14876,6,3.0,0.75
1,15824,6,4.0,1.00
2,16599,6,3.0,0.75
3,16500,6,3.0,0.75
4,14975,6,3.0,0.75


In [7]:
ASAP_SCORE_RANGES = {
    1: (2, 12),
    2: (1, 6),
    3: (0, 3),
    4: (0, 3),
    5: (0, 4),
    6: (0, 4),
    7: (0, 30),
    8: (0, 60),
}

Y_MIN, Y_MAX = ASAP_SCORE_RANGES[ESSAY_SET]
Y_MIN, Y_MAX

(0, 4)

In [8]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    spearman = pd.Series(y_true).corr(pd.Series(y_pred), method="spearman")

    return {
        "QWK": float(qwk),
        "MAE": float(mae),
        "RMSE": float(rmse),
        "Spearman": float(spearman) if not pd.isna(spearman) else np.nan,
    }


In [ ]:
def sample_pairs(df, num_pairs=1000, seed=42, min_per_essay=3):
    rng = np.random.default_rng(seed)
    essay_ids = df["essay_id"].astype(int).tolist()

    pairs = set()
    counts = {eid: 0 for eid in essay_ids}

    # 1. Ensure each essay appears at least min_per_essay times if possible
    attempts = 0
    max_attempts = num_pairs * 200

    while min(counts.values()) < min_per_essay and len(pairs) < num_pairs and attempts < max_attempts:
        underrepresented = [eid for eid, c in counts.items() if c < min_per_essay]

        a = int(rng.choice(underrepresented))
        b = int(rng.choice([eid for eid in essay_ids if eid != a]))

        x, y = sorted((a, b))

        if (x, y) not in pairs:
            pairs.add((x, y))
            counts[x] += 1
            counts[y] += 1

        attempts += 1

    # 2. Fill remaining pairs randomly
    while len(pairs) < num_pairs:
        a, b = rng.choice(essay_ids, size=2, replace=False)
        x, y = sorted((int(a), int(b)))
        pairs.add((x, y))

    return pd.DataFrame(list(pairs), columns=["essay_id_1", "essay_id_2"])


# Paper-like transductive setup:
# LCES main experiments score the target essay set as a whole. The scoring
# model is trained from comparisons among the essays to be scored, then the
# learned latent scores are linearly mapped to the rubric range.
p1_all_full = pd.concat(
    [
        p1_train.assign(split="train"),
        p1_val.assign(split="val"),
        p1_test.assign(split="test"),
    ],
    axis=0,
    ignore_index=True
)

# Paper says it randomly samples 10% essays from each ASAP prompt for evaluation.
# Use a fixed seed for reproducibility.
p1_all = (
    p1_all_full
    .sample(frac=0.10, random_state=SEED)
    .reset_index(drop=True)
)

M_PAIRWISE = 6500

all_pairs = sample_pairs(
    p1_all,
    num_pairs=M_PAIRWISE,
    seed=SEED,
    min_per_essay=20
)

print("P1 full:", p1_all_full.shape)
print("P1 paper-like 10% subset:", p1_all.shape)
print("Pairwise comparisons M:", all_pairs.shape)

used_ids = set(all_pairs["essay_id_1"]) | set(all_pairs["essay_id_2"])
print("Essays appearing in pairs:", len(used_ids))
print("Avg appearances per essay:", 2 * len(all_pairs) / len(p1_all))

all_pairs.head()


P1 all: (1800, 6)
Pairwise comparisons M: (6500, 2)


,essay_id_1,essay_id_2
0,15441,15715
1,15281,16251
2,16172,16459
3,15554,15592
4,14993,15030


In [10]:
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
# Có thể dùng v0.2 nếu v0.3 lỗi access/package:
# MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

# Rất quan trọng với decoder-only model
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"

# Mistral thường không có pad_token riêng.
# Cách ổn nhất là dùng eos_token làm pad_token.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

llm.config.pad_token_id = tokenizer.pad_token_id
llm.generation_config.pad_token_id = tokenizer.pad_token_id
llm.eval()

print("Loaded:", MODEL_ID)
print("padding_side:", tokenizer.padding_side)
print("pad_token:", tokenizer.pad_token)
print("pad_token_id:", tokenizer.pad_token_id)
print("eos_token:", tokenizer.eos_token)
print("eos_token_id:", tokenizer.eos_token_id)
print("model pad_token_id:", llm.config.pad_token_id)
print("generation pad_token_id:", llm.generation_config.pad_token_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: mistralai/Mistral-7B-Instruct-v0.3
padding_side: left
pad_token: </s>
pad_token_id: 2
eos_token: </s>
eos_token_id: 2
model pad_token_id: 2
generation pad_token_id: 2


In [11]:
P1_RUBRIC_COMPACT = """
Evaluate the quality of a source-dependent response to the excerpt "The Mooring Mast."

Task:
The student must describe the obstacles the builders of the Empire State Building faced in attempting to allow dirigibles to dock there. The response should support the answer with relevant and specific information from the excerpt.

A strong response should identify the major obstacles clearly and explain why those obstacles made docking dirigibles on top of the Empire State Building difficult, dangerous, impractical, or impossible.

Score Point 4:
Prefer the response that gives a clear, complete, and accurate description of the obstacles the builders faced in attempting to allow dirigibles to dock at the Empire State Building.
The response includes relevant and specific information from the excerpt.
It may explain several important obstacles, such as the stress a thousand-foot dirigible and wind pressure would put on the building's frame, the need to modify and strengthen the steel framework, the danger of hydrogen-filled dirigibles, the violent and shifting winds at the top of the building, the unsafe problem of a dirigible swiveling around the mast or needing weights above pedestrians, the law against airships flying too low over urban areas, and the failed attempts by the Los Angeles or the Goodyear blimp Columbia.
The response clearly connects these details to the difficulty of safely docking airships.

Score Point 3:
Prefer the response that gives a mostly clear, complete, and accurate description of the obstacles.
The response includes relevant information from the excerpt, but the support may be more general, less specific, or less complete than a Score Point 4 response.
It may identify several correct obstacles, such as wind, safety, building stress, or legal restrictions, but may not explain all of them fully or may not connect every detail clearly to the docking problem.

Score Point 2:
The response gives a partial description of the obstacles.
It includes limited information from the excerpt and may include some misinterpretations.
It may mention only one or two obstacles, such as wind or safety, but does not fully explain why they mattered.
The response may rely on vague statements, incomplete evidence, or mostly literal details without clearly explaining the problem of docking dirigibles.

Score Point 1:
The response gives a minimal description of the obstacles.
It includes little or no relevant information from the excerpt and may include misinterpretations.
It may only relate minimally to the task, such as saying that the building was tall, the airships were hard to dock, or the plan did not work, without meaningful explanation or support.

Score Point 0:
The response is totally incorrect, irrelevant, blank, or contains insufficient evidence to demonstrate comprehension of the excerpt or the question.

When comparing two responses, prefer the one that:
- addresses the question more directly,
- more accurately describes the obstacles to docking dirigibles,
- includes more of the key obstacles from the excerpt,
- better explains why those obstacles made docking difficult, dangerous, impractical, or illegal,
- uses more relevant and specific evidence from the excerpt,
- shows stronger comprehension of both explicit and implied information,
- is clearer, better organized, and easier to understand.

Key obstacles from the excerpt may include:
- the building's frame had to handle the stress of a large dirigible and wind pressure,
- the steel framework needed expensive modifications and strengthening,
- hydrogen-filled dirigibles created serious fire and safety risks,
- strong and constantly shifting winds at the top of the building made docking dangerous,
- a tethered dirigible could swivel around the mast,
- weights used to control dirigibles would be unsafe above pedestrians,
- laws prevented airships from flying too low over urban areas,
- actual docking attempts failed or were only limited publicity stunts,
- complete mooring equipment was never successfully installed or used as intended.

Choose "tie" only if the two responses are genuinely very close in quality.
""".strip()

In [12]:
P1_PROMPT_TEXT = """
The Mooring Mast
by Marcia Amidon Lüsted

When the Empire State Building was conceived, it was planned as the world's tallest building, taller even than the new Chrysler Building that was being constructed at Forty-second Street and Lexington Avenue in New York. At seventy-seven stories, the Chrysler Building was the tallest building before the Empire State Building began construction, and Al Smith was determined to outstrip it in height.

The architect building the Chrysler Building secretly constructed a 185-foot spire inside the building and then shocked the public and the media by hoisting it to the top of the Chrysler Building, bringing it to a height of 1,046 feet. Al Smith realized that he was close to losing the title of world's tallest building, so he announced that the Empire State Building would reach 1,250 feet. The top of the Empire State Building would serve a higher calling: it would be equipped for an age of transportation that aviation pioneers dreamed about. The building would have a mooring mast at its top for docking dirigibles, or zeppelins, which would carry passengers on transatlantic routes and future routes.

By the 1920s, dirigibles were being hailed as the transportation of the future. They were enormous steel-framed balloons filled with hydrogen or helium to make them lighter than air. Unlike balloons, dirigibles could be maneuvered with propellers and rudders, and passengers could ride in an enclosed gondola underneath. Dirigibles could travel for thousands of miles without refueling, but New York City lacked a suitable landing area. Al Smith saw an opportunity: a mooring mast on top of the Empire State Building would let dirigibles anchor for refueling, service, and passenger loading.

Dirigibles were docked by means of an electric winch, which hauled in a line from the front of the ship and tied it to a mast. The body of the dirigible could swing in the breeze, while passengers would walk down a gangplank to an observation platform. Architects and engineers consulted experts, toured mooring operations at the U.S. Naval Air Station in Lakehurst, New Jersey, and discussed practical and safe ways to moor airships to the mast.

Designing the mast was difficult. A thousand-foot dirigible moored to the top of the building by a single cable would add stress to the building's frame. The stress of the dirigible's load and wind pressure would have to be transmitted all the way down to the building's foundation. The steel frame of the Empire State Building had to be modified and strengthened, requiring over sixty thousand dollars' worth of changes.

The architects designed a shiny glass and stainless steel tower that imitated the shape of the building. It included observation areas, elevators, stairs, and spaces for baggage and tickets. The open observation platform on the 102nd floor was intended to double as the boarding area for dirigible passengers.

However, the mooring mast never fulfilled its purpose. One reason was safety: many dirigibles used hydrogen, which is highly flammable. The Hindenburg disaster later showed how dangerous a hydrogen airship accident could be above a densely populated city. Another obstacle was nature itself. Winds at the top of the building shifted constantly because of violent air currents. A tethered dirigible could swivel around the mast, and weighting it down would be unsafe because weights would dangle high above pedestrians.

There was also a legal problem: an existing law prevented airships from flying too low over urban areas, making it illegal for ships to tie up to or even approach the building. Two dirigibles attempted to reach the mast, but they could not dock safely. The U.S. Navy dirigible Los Angeles could not get close enough because of forceful winds. Later, the Goodyear blimp Columbia delivered newspapers by rope as a publicity stunt, but the idea of using the mast was soon shelved. By the late 1930s, dirigibles had given way to airplanes, and the rooms planned for dirigible passengers were converted for sightseers.

Prompt:
Based on the excerpt, describe the obstacles the builders of the Empire State Building faced in attempting to allow dirigibles to dock there. Support your answer with relevant and specific information from the excerpt.
""".strip()

In [13]:
SYSTEM_PROMPT_ASAP = """
You are an English teacher evaluating middle school exam essays.
Compare the two essays using the prompt and rubric.
Return only one valid JSON object.
Do not copy the essays.
Do not write anything outside the JSON.
""".strip()


def build_pairwise_prompt(essay1, essay2):
    return f"""
# Task
Which essay would receive a higher overall score?

# Prompt
{P1_PROMPT_TEXT}

# Rubric Guidelines
{P1_RUBRIC_COMPACT}

# Rules
- Do not penalize anonymized entities such as PERSON, ORGANIZATION, LOCATION, DATE, TIME, MONEY, PERCENT, CAPS, or NUM.
- Return only one valid JSON object.
- The value must be exactly one of: "essay1", "essay2", "tie".
- Do not explain.
- Do not copy the essays.

# Essay1
{essay1}

# Essay2
{essay2}

# Answer
Return your decision now as one valid JSON object only.
Use exactly this key: "preference".
Allowed values: "essay1", "essay2", "tie".

JSON:
""".strip()

In [14]:
def extract_preference(text):
    """Extract preference from JSON-like LLM output with robust fallbacks."""
    raw = "" if text is None else str(text)
    text = raw.strip()

    def normalize_pref(pref):
        pref = str(pref).lower().strip()
        pref = pref.replace(" ", "")
        pref = pref.replace("_", "")
        pref = pref.replace('"', "")
        pref = pref.replace("'", "")

        if pref in ["essay1", "1", "first", "firstessay"]:
            return "essay1"
        if pref in ["essay2", "2", "second", "secondessay"]:
            return "essay2"
        if pref in ["tie", "equal", "same", "samequality", "both", "neither"]:
            return "tie"
        return None

    def make_result(reasoning="", preference="tie", parse_failed=False):
        return {
            "reasoning": str(reasoning)[:500],
            "preference": preference,
            "raw_text": raw[:2000],
            "parse_failed": parse_failed,
        }

    # Empty output
    if text == "":
        return make_result("", "tie", parse_failed=True)

    # Remove markdown fences if model returns ```json ... ```
    cleaned = text
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned.strip())

    # 1. Direct JSON parse
    try:
        obj = json.loads(cleaned)
        if isinstance(obj, dict):
            pref = normalize_pref(obj.get("preference", ""))
            if pref is not None:
                return make_result(obj.get("reasoning", ""), pref, parse_failed=False)
    except Exception:
        pass

    # 2. Extract JSON-looking blocks and try each one
    json_blocks = re.findall(r"\{[^{}]*\}", cleaned, flags=re.DOTALL)
    for block in json_blocks:
        try:
            obj = json.loads(block)
            if isinstance(obj, dict):
                pref = normalize_pref(obj.get("preference", ""))
                if pref is not None:
                    return make_result(obj.get("reasoning", ""), pref, parse_failed=False)
        except Exception:
            continue

    # 3. Regex preference field from imperfect JSON/text
    pref_match = re.search(
        r'["\']?preference["\']?\s*[:=]\s*["\']?(essay\s*1|essay\s*2|essay1|essay2|tie|equal|same)',
        cleaned,
        flags=re.IGNORECASE,
    )
    if pref_match:
        pref = normalize_pref(pref_match.group(1))
        if pref is not None:
            return make_result(cleaned, pref, parse_failed=False)

    # 4. Final-decision style fallback
    decision_match = re.search(
        r'(decision|final decision|answer|winner)\s*[:=\-]?\s*["\']?(essay\s*1|essay\s*2|essay1|essay2|tie|equal|same)',
        cleaned,
        flags=re.IGNORECASE,
    )
    if decision_match:
        pref = normalize_pref(decision_match.group(2))
        if pref is not None:
            return make_result(cleaned, pref, parse_failed=False)

    # 5. Safer last-resort tail check
    # Only accept if exactly one of essay1 / essay2 appears in the tail.
    tail = cleaned[-500:].lower()

    has_essay1 = re.search(r"\bessay\s*1\b|\bessay1\b", tail) is not None
    has_essay2 = re.search(r"\bessay\s*2\b|\bessay2\b", tail) is not None
    has_tie = re.search(r"\btie\b|same score|equal quality|equally|same quality", tail) is not None

    if has_tie and not has_essay1 and not has_essay2:
        return make_result(cleaned, "tie", parse_failed=False)

    if has_essay1 and not has_essay2:
        return make_result(cleaned, "essay1", parse_failed=False)

    if has_essay2 and not has_essay1:
        return make_result(cleaned, "essay2", parse_failed=False)

    # True parser failure
    return make_result(cleaned, "tie", parse_failed=True)


tokenizer.padding_side = "left"

@torch.inference_mode()
def call_local_llm(prompts, max_new_tokens=32, temperature=0.0):
    """Batched local LLM inference for pairwise judgments."""
    input_texts = []

    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"

    for prompt in prompts:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT_ASAP},
            {"role": "user", "content": prompt},
        ]

        if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
            input_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            input_text = SYSTEM_PROMPT_ASAP + "\n\n" + prompt

        input_texts.append(input_text)

    inputs = tokenizer(
        input_texts,   # important: use chat-template text, not raw prompts
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,
    ).to(llm.device)

    do_sample = temperature is not None and temperature > 0

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "use_cache": True,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    if do_sample:
        generation_kwargs["temperature"] = temperature

    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            **generation_kwargs,
        )

    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[:, prompt_len:]

    texts = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )

    results = [extract_preference(t) for t in texts]

    del inputs, output_ids, generated_ids

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


In [15]:
# Quick smoke test
sample_essay_1 = p1_all.iloc[0]["essay"]
sample_essay_2 = p1_all.iloc[1]["essay"]

test_prompt = build_pairwise_prompt(sample_essay_1, sample_essay_2)
call_local_llm([test_prompt], max_new_tokens=64, temperature=0.1)


[{'reasoning': '',
  'preference': 'essay2',
  'raw_text': '{"preference": "essay2"}',
  'parse_failed': False}]

In [16]:
def preference_to_label(preference):
    pref = str(preference).lower().strip()
    pref = pref.replace(" ", "")
    pref = pref.replace("_", "")
    pref = pref.replace('"', "")
    pref = pref.replace("'", "")

    if pref in ["essay1", "1"]:
        return 1.0
    if pref in ["essay2", "2"]:
        return 0.0
    if pref in ["tie", "equal", "same", "both"]:
        return 0.5

    return 0.5

def debias_label(label_forward, label_reverse):
    """
    Forward: essay_id_1 as Essay 1, essay_id_2 as Essay 2.
    Reverse: essay_id_2 as Essay 1, essay_id_1 as Essay 2.

    If consistent:
      label_forward == 1 - label_reverse

    Else:
      tie.
    """
    if np.isclose(label_forward, 1.0 - label_reverse):
        return label_forward
    return 0.5

In [17]:
def generate_pairwise_judgments(
    df,
    pairs_df,
    debias=True,
    batch_size=4,
    max_new_tokens=32,
    temperature=0.0,
):
    """Generate pairwise LLM judgments with LCES position-bias correction.

    For Mistral 7B:
    - keep batch_size small
    - use deterministic decoding
    - retry empty generations once
    - convert reverse judgments back to original essay order
    """
    essay_map = df.set_index("essay_id")["essay"].to_dict()
    rows = []

    pair_rows = list(pairs_df.itertuples(index=False))

    def pref_to_original_label(pref, is_reverse=False):
        """
        Convert model preference to label in ORIGINAL pair direction.

        Original pair:
            label =  1 means essay_id_1 is better
            label = -1 means essay_id_2 is better
            label =  0 means tie

        Forward prompt:
            Essay1 = essay_id_1
            Essay2 = essay_id_2

        Reverse prompt:
            Essay1 = essay_id_2
            Essay2 = essay_id_1
        """
        pref = str(pref).strip().lower()

        if pref == "essay1":
            label = 1
        elif pref == "essay2":
            label = -1
        else:
            label = 0

        if is_reverse:
            label = -label

        return label

    def combine_forward_reverse(label_forward, label_reverse_original):
        """
        Conservative debiasing:
        - if forward and reverse agree after converting to original direction, keep label
        - otherwise mark as tie
        """
        if label_forward == label_reverse_original:
            return label_forward
        return 0

    def retry_empty_results(prompts, results):
        """
        Mistral sometimes returns empty raw_text.
        Retry only those failed prompts once.
        """
        failed_indices = [
            i for i, r in enumerate(results)
            if str(r.get("raw_text", "")).strip() == ""
        ]

        if len(failed_indices) == 0:
            return results

        retry_prompts = [prompts[i] for i in failed_indices]

        retry_results = call_local_llm(
            retry_prompts,
            max_new_tokens=max(max_new_tokens, 64),
            temperature=0.0,
        )

        for idx, retry_result in zip(failed_indices, retry_results):
            results[idx] = retry_result

        return results

    for start in tqdm(range(0, len(pair_rows), batch_size)):
        batch = pair_rows[start:start + batch_size]

        forward_prompts = []
        reverse_prompts = []
        meta = []

        for row in batch:
            id1 = int(row.essay_id_1)
            id2 = int(row.essay_id_2)

            essay1 = essay_map[id1]
            essay2 = essay_map[id2]

            forward_prompts.append(build_pairwise_prompt(essay1, essay2))

            if debias:
                reverse_prompts.append(build_pairwise_prompt(essay2, essay1))

            meta.append((id1, id2))

        forward_results = call_local_llm(
            forward_prompts,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
        )

        forward_results = retry_empty_results(forward_prompts, forward_results)

        if debias:
            reverse_results = call_local_llm(
                reverse_prompts,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
            )

            reverse_results = retry_empty_results(reverse_prompts, reverse_results)
        else:
            reverse_results = [None] * len(forward_results)

        for (id1, id2), result_forward, result_reverse in zip(meta, forward_results, reverse_results):
            pref_forward = result_forward.get("preference", "tie")
            label_forward = pref_to_original_label(pref_forward, is_reverse=False)

            raw_forward = result_forward.get("raw_text", "")
            failed_forward = str(raw_forward).strip() == ""

            if debias:
                pref_reverse = result_reverse.get("preference", "tie")
                label_reverse_original = pref_to_original_label(pref_reverse, is_reverse=True)

                raw_reverse = result_reverse.get("raw_text", "")
                failed_reverse = str(raw_reverse).strip() == ""

                final_label = combine_forward_reverse(
                    label_forward,
                    label_reverse_original,
                )

                rows.append({
                    "essay_id_1": id1,
                    "essay_id_2": id2,
                    "label": final_label,

                    "pref_forward": pref_forward,
                    "pref_reverse": pref_reverse,

                    "label_forward_original": label_forward,
                    "label_reverse_original": label_reverse_original,

                    "reason_forward": result_forward.get("reasoning", ""),
                    "reason_reverse": result_reverse.get("reasoning", ""),

                    "raw_forward": raw_forward,
                    "raw_reverse": raw_reverse,

                    "failed_forward": failed_forward,
                    "failed_reverse": failed_reverse,
                })

            else:
                rows.append({
                    "essay_id_1": id1,
                    "essay_id_2": id2,
                    "label": label_forward,

                    "pref_forward": pref_forward,
                    "pref_reverse": None,

                    "label_forward_original": label_forward,
                    "label_reverse_original": None,

                    "reason_forward": result_forward.get("reasoning", ""),
                    "reason_reverse": None,

                    "raw_forward": raw_forward,
                    "raw_reverse": None,

                    "failed_forward": failed_forward,
                    "failed_reverse": None,
                })

        del batch, forward_prompts, reverse_prompts, forward_results, reverse_results, meta

        batch_idx = start // batch_size
        if torch.cuda.is_available() and batch_idx % 10 == 0:
            torch.cuda.empty_cache()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame(rows)


def inspect_judgments(judgments):
    print("pref_forward")
    print(judgments["pref_forward"].value_counts(dropna=False))

    if "pref_reverse" in judgments.columns:
        print("\npref_reverse")
        print(judgments["pref_reverse"].value_counts(dropna=False))

    print("\nlabel")
    print(judgments["label"].value_counts(dropna=False))

    tie_rate = (judgments["label"] == 0).mean()
    print(f"\nTie rate: {tie_rate:.2%}")

    if "failed_forward" in judgments.columns:
        print("\nFailure rate:")
        print(judgments[["failed_forward", "failed_reverse"]].mean())

    if tie_rate > 0.60:
        raise ValueError("Tie rate is high. Inspect raw outputs before training RankNet.")


### Pairwise judgment generation
The following cell generates or loads cached LCES pairwise comparisons for the transductive Prompt 1 essay set.


In [18]:
JUDGMENT_CACHE = "all_judgments_p2_mistral7b_paperprompt_m6500_debias.csv"

if os.path.exists(JUDGMENT_CACHE):
    all_judgments = pd.read_csv(JUDGMENT_CACHE)
    print("Loaded cached judgments:", JUDGMENT_CACHE)
else:
    all_judgments = generate_pairwise_judgments(
        df=p1_all,
        pairs_df=all_pairs,
        debias=True,
        batch_size=64,
        max_new_tokens=32,
        temperature=0.0,
    )
    all_judgments.to_csv(JUDGMENT_CACHE, index=False)
    print("Saved judgments:", JUDGMENT_CACHE)

inspect_judgments(all_judgments)

# Free LLM VRAM before loading the embedding model and training RankNet.
if "llm" in globals():
    del llm
if "tokenizer" in globals():
    del tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()


  0%|          | 0/102 [00:00<?, ?it/s]

Saved judgments: all_judgments_p2_mistral7b_paperprompt_m6500_debias.csv
pref_forward
pref_forward
essay1    4675
essay2    1825
Name: count, dtype: int64

pref_reverse
pref_reverse
essay1    4557
essay2    1943
Name: count, dtype: int64

label
label
 0    2734
 1    1942
-1    1824
Name: count, dtype: int64

Tie rate: 42.06%

Failure rate:
failed_forward    0.0
failed_reverse    0.0
dtype: float64


In [19]:
used_ids = set(all_judgments["essay_id_1"]) | set(all_judgments["essay_id_2"])

print("Total P1 essays:", len(p1_all))
print("Essays appearing in comparisons:", len(used_ids))
print("Coverage:", len(used_ids) / len(p1_all))

pair_count = pd.concat([
    all_judgments["essay_id_1"],
    all_judgments["essay_id_2"],
]).value_counts()

print("\nPair count per essay:")
print(pair_count.describe())


Total P1 essays: 1800
Essays appearing in comparisons: 1800
Coverage: 1.0

Pair count per essay:
count    1800.000000
mean        7.222222
std         1.853307
min         4.000000
25%         6.000000
50%         7.000000
75%         8.000000
max        18.000000
Name: count, dtype: float64


In [20]:
# Paper main experiments use OpenAI text-embedding-3-large. To keep this
# notebook runnable without an API key, we use RoBERTa-base [CLS], which the
# paper also reports as a robust alternative in Appendix B.
EMBEDDING_MODEL = "roberta-base"

embedding_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
embedding_model = AutoModel.from_pretrained(EMBEDDING_MODEL).to(DEVICE)
embedding_model.eval()

@torch.inference_mode()
def encode_essays_cls(df, text_col="essay", batch_size=16, max_length=512):
    all_vecs = []
    texts = df[text_col].tolist()

    for start in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[start:start + batch_size]

        inputs = embedding_tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        ).to(DEVICE)

        outputs = embedding_model(**inputs)
        cls_vec = outputs.last_hidden_state[:, 0, :]
        all_vecs.append(cls_vec.detach().cpu().numpy())

        del inputs, outputs, cls_vec

    return np.vstack(all_vecs).astype(np.float32)


all_embeddings = encode_essays_cls(p1_all, batch_size=16, max_length=512)

print("Embedding model:", EMBEDDING_MODEL)
print("all_embeddings:", all_embeddings.shape)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  0%|          | 0/113 [00:00<?, ?it/s]

Embedding model: roberta-base
all_embeddings: (1800, 768)


In [21]:
class RankNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, dropout=0.3):
        super().__init__()

        self.scorer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.scorer(x).squeeze(-1)

In [22]:
def train_ranknet(
    df,
    embeddings,
    judgments,
    epochs=100,
    batch_size=4096,
    lr=1e-3,
    weight_decay=0.01,
    hidden_dim=256,
    dropout=0.3,
    device=DEVICE
):
    essay_ids = df["essay_id"].astype(int).tolist()
    id_to_idx = {eid: idx for idx, eid in enumerate(essay_ids)}

    pair_df = judgments[
        judgments["essay_id_1"].isin(id_to_idx) &
        judgments["essay_id_2"].isin(id_to_idx)
    ].copy()

    assert len(pair_df) > 0, "No valid pairwise judgments for this dataframe."

    idx_i = pair_df["essay_id_1"].map(id_to_idx).values
    idx_j = pair_df["essay_id_2"].map(id_to_idx).values
    labels = pair_df["label"].map({
        1: 1.0,
        -1: 0.0,
        0: 0.5,
    }).values.astype(np.float32)
    x = torch.tensor(embeddings, dtype=torch.float32, device=device)

    model = RankNet(
        input_dim=embeddings.shape[1],
        hidden_dim=hidden_dim,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    loss_fn = nn.BCELoss()

    n = len(pair_df)

    for epoch in range(1, epochs + 1):
        order = np.random.permutation(n)
        epoch_losses = []

        model.train()

        for start in range(0, n, batch_size):
            batch = order[start:start + batch_size]

            bi = torch.tensor(idx_i[batch], dtype=torch.long, device=device)
            bj = torch.tensor(idx_j[batch], dtype=torch.long, device=device)
            y = torch.tensor(labels[batch], dtype=torch.float32, device=device)

            s_i = model(x[bi])
            s_j = model(x[bj])

            pred = torch.sigmoid(s_i - s_j)
            loss = loss_fn(pred, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

        if epoch == 1 or epoch % 10 == 0:
            print(f"Epoch {epoch:03d} | loss = {np.mean(epoch_losses):.4f}")

    return model

In [23]:
# Paper hyperparameters: epochs=100, batch_size=4096, hidden_dim=256,
# dropout=0.3, weight_decay=0.01, lr=0.001.
ranknet = train_ranknet(
    df=p1_all,
    embeddings=all_embeddings,
    judgments=all_judgments,
    epochs=100,
    batch_size=4096,
    lr=1e-3,
    weight_decay=0.01,
    hidden_dim=256,
    dropout=0.3,
)


Epoch 001 | loss = 0.6921
Epoch 010 | loss = 0.6591
Epoch 020 | loss = 0.6094
Epoch 030 | loss = 0.5700
Epoch 040 | loss = 0.5436
Epoch 050 | loss = 0.5295
Epoch 060 | loss = 0.5191
Epoch 070 | loss = 0.5098
Epoch 080 | loss = 0.5058
Epoch 090 | loss = 0.5009
Epoch 100 | loss = 0.5007


In [24]:
@torch.no_grad()
def predict_latent_scores(model, embeddings, device=DEVICE):
    model.eval()
    x = torch.tensor(embeddings, dtype=torch.float32, device=device)
    scores = model(x).detach().cpu().numpy()
    return scores


all_latent = predict_latent_scores(ranknet, all_embeddings)

print("latent min:", all_latent.min())
print("latent max:", all_latent.max())
print("latent std:", all_latent.std())
print("latent head:", all_latent[:10])


latent min: -3.742299
latent max: 1.2046076
latent std: 0.8263685
latent head: [ 0.15612596  0.5204316   0.05523346 -0.3888293  -0.8188547  -1.0852425
 -0.556725   -0.04526908 -0.19272673  0.24544096]


In [25]:
def fit_latent_mapper(latent_scores_ref, y_min, y_max):
    """Paper's linear latent-score conversion to the rubric range."""
    s_min = float(np.min(latent_scores_ref))
    s_max = float(np.max(latent_scores_ref))

    def mapper(latent_scores):
        latent_scores = np.asarray(latent_scores, dtype=float)

        if np.isclose(s_min, s_max):
            mapped = np.full_like(latent_scores, fill_value=(y_min + y_max) / 2)
        else:
            mapped = (latent_scores - s_min) / (s_max - s_min)
            mapped = mapped * (y_max - y_min) + y_min

        mapped = np.rint(mapped)
        mapped = np.clip(mapped, y_min, y_max)

        return mapped.astype(int)

    return mapper


score_mapper = fit_latent_mapper(all_latent, Y_MIN, Y_MAX)
all_pred = score_mapper(all_latent)

p1_all_scored = p1_all[[
    "essay_id",
    "split",
    "essay_set",
    "essay",
    "domain1_score",
    "domain1_score_norm",
]].copy()

p1_all_scored["latent_score"] = all_latent
p1_all_scored["pred_score"] = all_pred
p1_all_scored["abs_error"] = np.abs(
    p1_all_scored["domain1_score"] - p1_all_scored["pred_score"]
)

print(pd.Series(all_pred).value_counts().sort_index())
p1_all_scored.head()


0     11
1    157
2    657
3    923
4     52
Name: count, dtype: int64


,essay_id,split,essay_set,essay,domain1_score,domain1_score_norm,latent_score,pred_score,abs_error
0,14876,train,6,In the excerpt The Mooring Mast by Marcia Amid...,3.0,0.75,0.156126,3,0.0
1,15824,train,6,The idea of using a mooring mast in order to a...,4.0,1.00,0.520432,3,1.0
2,16599,train,6,"Based on the excerpt, the builders of the Empi...",3.0,0.75,0.055233,3,0.0
3,16500,train,6,When building the dock for dirigibles. Al Smit...,3.0,0.75,-0.388829,3,0.0
4,14975,train,6,"According to the excerpt ""The Mooring Mast"" by...",3.0,0.75,-0.818855,2,1.0


In [26]:
metrics_rows = []

for split in ["train", "val", "test", "all"]:
    if split == "all":
        df_eval = p1_all_scored
    else:
        df_eval = p1_all_scored[p1_all_scored["split"] == split]

    metrics_rows.append({
        "split": split,
        **compute_metrics(
            y_true=df_eval["domain1_score"],
            y_pred=df_eval["pred_score"],
        ),
    })

pd.DataFrame(metrics_rows)


,split,QWK,MAE,RMSE,Spearman
0,train,0.620104,0.521808,0.759815,0.642895
1,val,0.628068,0.505556,0.741620,0.656172
2,test,0.602671,0.548747,0.784601,0.643427
3,all,0.617362,0.525556,0.763035,0.644032


In [27]:
p1_test_results = p1_all_scored[p1_all_scored["split"] == "test"].copy()

p1_test_results[[
    "essay_id",
    "domain1_score",
    "pred_score",
    "abs_error",
    "latent_score",
]].head(20)


,essay_id,domain1_score,pred_score,abs_error,latent_score
1441,15721,4.0,3,1.0,0.157878
1442,15226,4.0,3,1.0,0.009059
1443,15716,2.0,2,0.0,-1.627120
1444,15504,1.0,2,1.0,-1.766116
1445,15814,4.0,3,1.0,0.582774
1446,15076,3.0,3,0.0,-0.516580
1447,15264,4.0,3,1.0,-0.206690
1448,14846,4.0,3,1.0,-0.441015
1449,16547,3.0,3,0.0,0.567956
1450,14921,3.0,3,0.0,-0.454521


In [28]:
# Worst predictions on the held-out test split
p1_test_results.sort_values("abs_error", ascending=False)[[
    "essay_id",
    "domain1_score",
    "pred_score",
    "abs_error",
    "latent_score",
    "essay",
]].head(10)


,essay_id,domain1_score,pred_score,abs_error,latent_score,essay
1754,15656,3.0,1,2.0,-1.982928,The builders of the empire state building face...
1749,16113,0.0,2,2.0,-1.594423,"In the excerpt called ""The Mooring Mast, by Ma..."
1745,16185,4.0,2,2.0,-0.974979,The builders of the Empire State building face...
1755,15836,1.0,3,2.0,-0.073693,Architects struggled with obstacles as they we...
1495,15097,4.0,2,2.0,-0.798358,"In the excerpt ""The Mooring @CAPS1"" by Marcia ..."
1540,15607,4.0,2,2.0,-0.805692,This dream of the aviation pioneers was travel...
1727,15598,3.0,1,2.0,-2.275216,"In this excerpt ""The Mooring Mast"", by @ORGANI..."
1489,15200,4.0,2,2.0,-1.017148,In the process of building the dock for dirigi...
1542,15972,0.0,2,2.0,-1.209726,Dirigibles had a top speed of eighty miles. So...
1767,15694,3.0,1,2.0,-2.556160,The problems That The empire state building fa...


In [29]:
from google.colab import files, runtime
import os
import time

JUDGMENT_CACHE = "all_judgments_p2_mistral7b_paperprompt_m6500_debias.csv"

possible_paths = [
    f"/content/{JUDGMENT_CACHE}",
    f"/content/output/{JUDGMENT_CACHE}",
]

SAVE_PATH = None
for path in possible_paths:
    if os.path.isfile(path):
        SAVE_PATH = path
        break

if SAVE_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy file ở:\n" + "\n".join(possible_paths)
    )

print(f"Đang tải: {SAVE_PATH}")
files.download(SAVE_PATH)



Đang tải: /content/all_judgments_p2_mistral7b_paperprompt_m6500_debias.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>